# 05 – OCR Evaluation (Colab / local CPU)

Evaluates CER/WER across 4 pipeline arms:
1. Raw photo
2. Perspective correction only
3. Perspective + classical baseline
4. Perspective + U-Net (gain-map)

Statistical tests: Wilcoxon signed-rank, bootstrap 95% CI.

In [ ]:
!pip -q install jiwer pytesseract easyocr scikit-image scipy pyyaml
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', 'tesseract-ocr'], check=True)

In [ ]:
import os, sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

COLAB = 'google.colab' in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRescue'
else:
    PROJECT_ROOT = '.'

sys.path.insert(0, PROJECT_ROOT)

with open(os.path.join(PROJECT_ROOT, 'configs/config.yaml')) as f:
    cfg = yaml.safe_load(f)

import torch
from src.models import build_model
from src.infer import load_checkpoint, gain_map_rescue
from src.geometry import correct_perspective
from src.classical import classical_pipeline
from src.ocr import run_ocr, normalize_text
from src.metrics import compute_cer, compute_wer, wilcoxon_test, bootstrap_ci, summarise

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
CKPT = os.path.join(PROJECT_ROOT, 'experiments/checkpoints/best.pth')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_model(cfg).to(device)
if Path(CKPT).exists():
    load_checkpoint(model, CKPT, device=device)
    print(f'Loaded checkpoint: {CKPT}')
else:
    print('WARNING: No checkpoint found. U-Net arm will use random weights.')
model.eval();

In [ ]:
# ── Pipeline arms ──────────────────────────────────────────────────────────
def arm_raw(img):
    return img

def arm_geom(img):
    result, _ = correct_perspective(img)
    return result

def arm_classic(img):
    result, _ = correct_perspective(img)
    return classical_pipeline(result)

def arm_unet(img):
    result, _ = correct_perspective(img)
    return gain_map_rescue(
        result, model, device=device,
        input_size=cfg['preprocessing']['image_size'],
        blur_ksize=cfg['inference']['gain_map_blur_ksize'],
        gain_min_clamp=cfg['inference']['gain_map_min_clamp'],
    )

ARMS = {
    'Raw':      arm_raw,
    'Geometry': arm_geom,
    'Classical': arm_classic,
    'U-Net':    arm_unet,
}

In [ ]:
# ── SmartDoc-QA evaluation ─────────────────────────────────────────────────
from src.data import SmartDocQA
from tqdm.auto import tqdm
from joblib import Parallel, delayed

smartdoc_root = os.path.join(PROJECT_ROOT, 'data', 'smartdoc_qa')
N_SAMPLE = 300   # start with 300; scale up if time allows

dataset = SmartDocQA(smartdoc_root, n_sample=N_SAMPLE, seed=42)
print(f'SmartDoc-QA samples: {len(dataset)}')

OCR_ENGINE = 'tesseract'

def evaluate_record(rec):
    img = np.array(Image.open(rec['path']).convert('RGB'))
    gt  = normalize_text(rec['gt_text'])
    result = {'doc_id': rec['doc_id'], 'capture': rec['capture']}
    for arm_name, arm_fn in ARMS.items():
        processed = arm_fn(img)
        hyp = normalize_text(run_ocr(processed, engine=OCR_ENGINE))
        result[f'cer_{arm_name}'] = compute_cer(gt, hyp)
        result[f'wer_{arm_name}'] = compute_wer(gt, hyp)
    return result

records = dataset.get_records()
results = Parallel(n_jobs=cfg['ocr']['n_jobs'], prefer='threads')(
    delayed(evaluate_record)(r) for r in tqdm(records, desc='OCR evaluation')
)

df = pd.DataFrame(results)
print(df.head())

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
summary_rows = []
for arm in ARMS:
    cer_col = f'cer_{arm}'
    wer_col = f'wer_{arm}'
    s = summarise(df[cer_col].tolist(), label=arm)
    s['mean_wer'] = df[wer_col].mean()
    summary_rows.append(s)

summary_df = pd.DataFrame(summary_rows)
print('\n── CER Summary ─────────────────────────────────────────────────────')
print(summary_df[['label', 'n', 'mean', 'std', 'ci95_lo', 'ci95_hi', 'mean_wer']]
      .rename(columns={'mean': 'mean_cer'})
      .to_string(index=False))

os.makedirs(os.path.join(PROJECT_ROOT, 'results', 'tables'), exist_ok=True)
summary_df.to_csv(os.path.join(PROJECT_ROOT, 'results', 'tables', 'ocr_smartdoc.csv'), index=False)

In [ ]:
# ── Statistical tests ──────────────────────────────────────────────────────
print('Wilcoxon test: Raw vs U-Net CER')
stat, p = wilcoxon_test(df['cer_Raw'].tolist(), df['cer_U-Net'].tolist(), alternative='greater')
print(f'  Statistic={stat:.1f}, p={p:.4f}')

print('\nWilcoxon test: Classical vs U-Net CER')
stat, p = wilcoxon_test(df['cer_Classical'].tolist(), df['cer_U-Net'].tolist(), alternative='greater')
print(f'  Statistic={stat:.1f}, p={p:.4f}')

In [ ]:
# ── Bar chart ──────────────────────────────────────────────────────────────
arms  = [r['label'] for r in summary_rows]
means = [r['mean']  for r in summary_rows]
lo    = [r['mean'] - r['ci95_lo'] for r in summary_rows]
hi    = [r['ci95_hi'] - r['mean'] for r in summary_rows]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(arms, means, yerr=[lo, hi], capsize=6,
              color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'], alpha=0.85)
ax.set_ylabel('Mean CER (lower is better)')
ax.set_title('OCR Character Error Rate by Pipeline Arm (SmartDoc-QA)')
ax.set_ylim(0, max(means) * 1.3)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.005, f'{m:.3f}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'ocr_cer_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── Cross-engine validation: EasyOCR ───────────────────────────────────────
# Re-run U-Net arm with EasyOCR to confirm finding is not Tesseract-specific
# (This takes longer; run when time permits)

def evaluate_unet_easyocr(rec):
    img = np.array(Image.open(rec['path']).convert('RGB'))
    gt  = normalize_text(rec['gt_text'])
    processed = arm_unet(img)
    hyp = normalize_text(run_ocr(processed, engine='easyocr'))
    return {'cer_unet_easy': compute_cer(gt, hyp)}

print('Running EasyOCR cross-validation on 50 samples …')
subsample = records[:50]
easy_results = Parallel(n_jobs=2, prefer='threads')(
    delayed(evaluate_unet_easyocr)(r) for r in tqdm(subsample, desc='EasyOCR')
)
easy_cer = [r['cer_unet_easy'] for r in easy_results]
tess_cer = df['cer_U-Net'].tolist()[:50]
print(f'Tesseract U-Net CER: {np.mean(tess_cer):.3f}')
print(f'EasyOCR  U-Net CER: {np.mean(easy_cer):.3f}')